In [1]:
import os
import json
import pickle
import pandas as pd
from pathlib import Path
from elasticsearch import Elasticsearch

In [3]:
import re
import pandas as pd
from pathlib import Path
import pickle
from tqdm import tqdm

# 1. กำหนดคอลัมน์ที่ต้องใช้
SEARCH_COLS = ['Name', 'RecipeIngredientParts', 'RecipeInstructions']
TAG_COLS    = ['Keywords', 'RecipeCategory']
RAW_COLS    = ['RecipeId', 'Images', 'Description', 'RecipeIngredientQuantities',
               'AggregatedRating', 'ReviewCount', 'TotalTime', 'RecipeYield']
ALL_COLS    = SEARCH_COLS + TAG_COLS + RAW_COLS

def get_and_clean_data_for_es() -> pd.DataFrame:
    csv_path = '../resources/recipes_updated.csv'
    cache_path = Path('../resources/cleaned_ready_for_es.pkl')

    if cache_path.exists():
        print("Loading cleaned data from the cache...")
        with open(cache_path, 'rb') as f:
            return pickle.load(f)

    print(f"Data loading from {csv_path}...")
    df = pd.read_csv(csv_path, usecols=lambda c: c in ALL_COLS)
    df = df.dropna(subset=['Name', 'RecipeIngredientParts', 'RecipeInstructions'])
    df = df.drop_duplicates(subset=['RecipeId']).reset_index(drop=True)

    print("Sanitizing text for search (SEARCH_COLS)....")
    for col in tqdm(SEARCH_COLS, desc="Cleaning Search Text", unit="col"):
        df[col + '_clean'] = (
            df[col]
            .astype(str)
            .str.lower()
            .str.replace(r'[^a-z0-9\s]', ' ', regex=True)
            .str.replace(r'\s+', ' ', regex=True)
            .str.strip()
        )

    def clean_r_format_to_list(val):
        if pd.isna(val) or val == 'character(0)' or str(val).strip() == '':
            return []
        val_str = str(val).replace('\\u003C', '<').replace('\\"', '')
        items = re.findall(r'"([^"]+)"', val_str)
        return items if items else [val_str.strip()]

    # แปลงเวลา PT1H30M -> 90 minutes
    def parse_pt_time(pt_str):
        if pd.isna(pt_str) or not isinstance(pt_str, str):
            return "0 minutes"
        val = pt_str.replace("PT", "")
        h = re.search(r'(\d+)H', val)
        m = re.search(r'(\d+)M', val)
        hours = int(h.group(1)) if h else 0
        mins = int(m.group(1)) if m else 0
        total_mins = hours * 60 + mins
        return f"{total_mins} minutes"

    print("Removing R format and converting to an array...")
    cols_to_list = ['RecipeIngredientQuantities', 'RecipeIngredientParts', 'Keywords']
    for col in tqdm(cols_to_list, desc="Converting to Lists", unit="col"):
        df[col] = df[col].apply(clean_r_format_to_list)


    df['TotalTime'] = df['TotalTime'].apply(parse_pt_time)

    null_fill_map = {
        'Images': '', 'Description': '', 'RecipeCategory': 'Uncategorized',
        'AggregatedRating': 0.0, 'ReviewCount': 0
    }
    for col, fill_val in null_fill_map.items():
        df[col] = df[col].fillna(fill_val)

    print(f"\nsave to Cache  {cache_path}")
    with open(cache_path, 'wb') as f:
        pickle.dump(df, f)

    return df

if __name__ == '__main__':
    df_clean = get_and_clean_data_for_es()
    print("\nExample of cleaned data:")
    print(df_clean[['Name_clean', 'RecipeIngredientParts_clean']].head())

Data loading from ../resources/recipes_updated.csv...
Sanitizing text for search (SEARCH_COLS)....


Cleaning Search Text: 100%|██████████| 3/3 [00:10<00:00,  3.41s/col]


Removing R format and converting to an array...


Converting to Lists: 100%|██████████| 3/3 [00:02<00:00,  1.09col/s]



save to Cache  ../resources/cleaned_ready_for_es.pkl

Example of cleaned data:
                          Name_clean  \
0  low fat berry blue frozen dessert   
1                            biryani   
2                      best lemonade   
3     carina s tofu vegetable kebabs   
4                       cabbage soup   

                         RecipeIngredientParts_clean  
0  blueberries granulated sugar vanilla yogurt le...  
1  saffron milk hot green chili peppers onions ga...  
2  sugar lemons rind of lemon zest of fresh water...  
3  extra firm tofu eggplant zucchini mushrooms so...  
4    plain tomato juice cabbage onion carrots celery  


In [4]:
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200")
es.info().body

{'name': 'ff82cb471847',
 'cluster_name': 'docker-cluster',
 'cluster_uuid': 'ijVw39HaS2OPK1ErLjux0Q',
 'version': {'number': '9.2.4',
  'build_flavor': 'default',
  'build_type': 'docker',
  'build_hash': 'dfc5c38614c29a598e132c035b66160d3d350894',
  'build_date': '2026-01-08T22:07:25.170027027Z',
  'build_snapshot': False,
  'lucene_version': '10.3.2',
  'minimum_wire_compatibility_version': '8.19.0',
  'minimum_index_compatibility_version': '8.0.0'},
 'tagline': 'You Know, for Search'}

In [6]:
import pandas as pd
import pickle
from pathlib import Path
from elasticsearch import Elasticsearch, helpers
import numpy as np
from tqdm import tqdm

class CyosojvpIndexer:
    def __init__(self):
        self.data_path = Path('../resources/cleaned_ready_for_es.pkl')
        print(f"Data loading from {self.data_path} ...")
        with open(self.data_path, 'rb') as f:
            self.df = pickle.load(f)

        self.es_client = Elasticsearch(
            "http://localhost:9200",
        )
        self.index_name = 'cyosojvp_recipes'

    def create_index_with_mapping(self):
        self.es_client.options(ignore_status=[400, 404]).indices.delete(index=self.index_name)

        mapping = {
            "settings": {
                "number_of_shards": 1,
                "number_of_replicas": 0,
                "analysis": {
                    "analyzer": {
                        "english_analyzer": {
                            "type": "english"
                        }
                    }
                }
            },
            "mappings": {
                "properties": {
                    "RecipeId": {"type": "keyword"},
                    "Name_clean": {
                        "type": "text",
                        "analyzer": "english_analyzer",
                        "fields": {
                            "keyword": {"type": "keyword"}
                        }
                    },
                    "RecipeIngredientParts_clean": {"type": "text", "analyzer": "english_analyzer"},
                    "RecipeInstructions_clean": {"type": "text", "analyzer": "english_analyzer"},
                    "RecipeCategory": {"type": "keyword"},
                    "Keywords": {"type": "text"},
                    "Images": {"type": "keyword", "index": False},
                    "Description": {"type": "text", "index": False},
                    "RecipeIngredientQuantities": {"type": "text", "index": False},
                    "AggregatedRating": {"type": "float"},
                    "ReviewCount": {"type": "integer"},
                    "TotalTime": {"type": "keyword"}
                }
            }
        }

        self.es_client.indices.create(index=self.index_name, body=mapping)
        print(f"Create Index '{self.index_name}' and Analyzed")

    def generate_actions(self):
        df_clean = self.df.replace({np.nan: None})
        records = df_clean.to_dict(orient='records')

        for row in tqdm(records, desc="⚡ Uploading to Elasticsearch", unit=" menu", colour="green"):
            yield {
                "_index": self.index_name,
                "_id": str(row["RecipeId"]),
                "_source": row
            }

    def run_indexer(self):
        if not self.es_client.ping():
            print("Unable to connect to Elasticsearch. Please check your server and password.")
            return

        print("Preparing to create an index...")
        self.create_index_with_mapping()

        print(f"Begin the data submission process. {len(self.df):,} menu...")

        success, failed = helpers.bulk(
            self.es_client,
            self.generate_actions(),
            chunk_size=2000,
            stats_only=True,
            raise_on_error=False
        )

        print(f"Completed! Database access successful {success:,} items | Failed {failed} items")

if __name__ == "__main__":
    indexer = CyosojvpIndexer()
    indexer.run_indexer()

Data loading from ../resources/cleaned_ready_for_es.pkl ...
Preparing to create an index...
Create Index 'cyosojvp_recipes' and Analyzed
Begin the data submission process. 520,297 menu...


⚡ Uploading to Elasticsearch: 100%|██████████| 520297/520297 [01:00<00:00, 8640.09 menu/s] 


Completed! Database access successful 520,297 items | Failed 0 items


In [7]:
print(es.count(index="cyosojvp_recipes"))

{'count': 520297, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}}


In [8]:
from elasticsearch import Elasticsearch
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

INDEX_NAME = "cyosojvp_recipes"

es = Elasticsearch(
    "http://localhost:9200"
)

def search_recipes(query):
    if not query:
        return "กรุณาใส่คำค้นหา (query)"

    body = {
        "size": 12,
        "query": {
            "multi_match": {
                "query": query,
                # 1. วางตัวคูณคะแนนตรงนี้ครับ ถึงจะถูกต้อง!
                "fields": [
                    "Name_clean^5",
                    "RecipeIngredientParts_clean^2",
                    "RecipeInstructions_clean"
                ],
                "type": "cross_fields",
                "operator": "and"
            }
        },
        "suggest": {
            "text": query,
            "spell_suggest": {
                "phrase": {
                    "field": "Name_clean",
                    "size": 1,
                    "gram_size": 3,
                    "direct_generator": [
                        {
                            # 2. แก้กลับเป็นของเดิม
                            "field": "Name_clean",
                            "suggest_mode": "always"
                        }
                    ]
                }
            }
        },
        # 3. โบนัส: เพิ่มระบบ Highlight (Score 14)
        "highlight": {
            "fields": {
                "RecipeInstructions_clean": {
                    "pre_tags": ["**"],  # คร่อมตัวหนา
                    "post_tags": ["**"],
                    "fragment_size": 100, # ตัดมาโชว์แค่ 100 ตัวอักษร
                    "number_of_fragments": 1
                }
            }
        }
    }

    response = es.search(index=INDEX_NAME, body=body)

    suggestions = []
    if "suggest" in response and response["suggest"]["spell_suggest"][0]["options"]:
        for option in response["suggest"]["spell_suggest"][0]["options"]:
            suggestions.append(option["text"])

    if suggestions:
        print(f"💡 Did you mean: {', '.join(suggestions)} ?\n")

    max_score = response["hits"]["max_score"]
    if not max_score:
        max_score = 1.0

    print(f"🔍 Results for '{query}':")
    print("="*60)

    hits = []
    for i, hit in enumerate(response["hits"]["hits"]):
        item = hit["_source"]
        score = round((hit["_score"] / max_score), 4)
        rank = i + 1

        ingredients = item.get("RecipeIngredientParts", [])
        if isinstance(ingredients, list):
            ingredients_str = ", ".join(ingredients)
        else:
            ingredients_str = str(ingredients).replace(" ", ", ")

        # 4. ดึง Highlight มาแสดงผล
        snippet = ""
        if "highlight" in hit and "RecipeInstructions_clean" in hit["highlight"]:
            snippet = hit["highlight"]["RecipeInstructions_clean"][0]

        display_text = (
            f"Rank {rank} | Score: {score}\n"
            f"Recipe name: {item.get('Name', 'Unknown')}\n"
            f"Total time: {item.get('TotalTime', 'Unknown')}\n"
            f"Category: {item.get('RecipeCategory', 'Uncategorized')}\n"
            f"Ingredients: {ingredients_str}\n"
        )
        # ถ้ามี Highlight ให้โชว์ด้วย
        if snippet:
            display_text += f"Snippet: ...{snippet}...\n"

        display_text += "-"*60

        print(display_text)

        item["Score"] = score
        item["Rank"] = rank
        hits.append(item)

    return hits


In [9]:
results = search_recipes("spicy chicken")

🔍 Results for 'spicy chicken':
Rank 1 | Score: 1.0
Recipe name: Spicy Chicken
Total time: 75 minutes
Category: Chicken Breast
Ingredients: onion garlic butter margarine chicken broth tomato sauce green onions Worcestershire sauce dried thyme cayenne pepper salt garlic powder pepper basil ground cumin boneless skinless chicken breasts angel hair pasta sour cream
Snippet: ...and toss to coat in a skillet saute **chicken** in remaining butter until juices run clear add **chicken** to...
------------------------------------------------------------
Rank 2 | Score: 1.0
Recipe name: Spicy Chicken
Total time: 16 minutes
Category: Chicken
Ingredients: boneless skinless chicken breast halves olive oil unsalted butter onion garlic clove dry white wine chipotle chile in adobo heavy cream adobo sauce
Snippet: ...put **chicken** breasts between pieces of waxed paper and pound to a thickness of about 1 2 inch heat large...
------------------------------------------------------------
Rank 3 | Score: 1